# 🏦 Automated Customer Complaint Classification System
## Exploratory Data Analysis, NLP Preprocessing, and Machine Learning Benchmarking
**B.Tech NLP Project-Based Learning (PBL)**

This notebook provides a step-by-step walkthrough of:
1. **Dataset Loading & Exploratory Data Analysis (EDA)**
2. **NLP Preprocessing Pipeline** (Tokenization, Stopword Removal, Lemmatization)
3. **TF-IDF Feature Extraction & N-gram Analysis**
4. **Multi-Model Training & Benchmarking** (LinearSVC, Logistic Regression, Multinomial NB, Random Forest)
5. **Performance Evaluation & Confusion Matrix Analysis**
6. **Inference Demonstration & Explainability**

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import nltk
import json
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('vader_lexicon', quiet=True)

print('All libraries imported successfully!')

### 1. Load and Explore the Complaint Dataset

In [ ]:
dataset_path = '../dataset/complaints.csv'
df = pd.read_csv(dataset_path)
print(f'Total Complaint Samples: {len(df)}')
print('\nClass Distribution:')
print(df['category'].value_counts())
df.head()

### 2. NLP Preprocessing Step-by-Step

In [ ]:
import sys
sys.path.append('..')
from src.preprocessing import clean_text, tokenize_text, remove_stopwords, lemmatize_tokens, preprocess_pipeline

sample_raw = df.iloc[0]['complaint_text']
print('--- Raw Input Text ---')
print(sample_raw)

cleaned = clean_text(sample_raw)
tokens = tokenize_text(cleaned)
filtered = remove_stopwords(tokens)
lemmas = lemmatize_tokens(filtered)

print('\n--- After Clean Text ---')
print(cleaned)
print('\n--- Tokens ---')
print(tokens[:10])
print('\n--- Filtered & Lemmatized ---')
print(lemmas[:10])
print('\n--- Full Pipeline Output ---')
print(preprocess_pipeline(sample_raw))

### 3. Feature Extraction (TF-IDF Vectorization)

In [ ]:
# Preprocess all dataset texts
df['cleaned_text'] = df['complaint_text'].apply(preprocess_pipeline)

# Vectorize with TF-IDF
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True)
X = tfidf.fit_transform(df['cleaned_text'])
y = df['category']

print(f'TF-IDF Feature Matrix Shape: {X.shape}')
print(f'Sample Vocabulary Terms: {list(tfidf.vocabulary_.keys())[:15]}')

### 4. Train-Test Split and Model Benchmarking

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    'LinearSVC': LinearSVC(random_state=42, max_iter=2000),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Multinomial NB': MultinomialNB(alpha=0.1),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    results.append({'Model': name, 'Accuracy': f'{acc*100:.2f}%', 'Macro F1': f'{f1:.4f}'})

results_df = pd.DataFrame(results)
results_df

### 5. Detailed Classification Report (Best Model)

In [ ]:
best_model = models['LinearSVC']
y_pred_best = best_model.predict(X_test)
print(classification_report(y_test, y_pred_best))

### 6. Interactive Prediction & Explainability Demo

In [ ]:
from src.predict import predict_complaint

sample_query = 'I noticed an unauthorized charge of $1,200 on my credit card. Please reverse it immediately and issue a replacement card.'
res = predict_complaint(sample_query)

print('=== PREDICTION RESULT ===')
print(f'Category:   {res["category"]}')
print(f'Confidence: {res["confidence_percentage"]}%')
print(f'Sentiment:  {res["sentiment"]} ({res["sentiment_score"]})')
print(f'Urgency:    {res["urgency"]}')
print(f'Department: {res["department"]}')
print(f'Keywords:   {res["keywords"]}')